In [ ]:
# -*- coding: utf-8 -*-
"""
Multi-Target Prediction Pipeline (IC50, CC50, SI)
Пайплайн для предсказания трёх параметров токсичности соединений:
- IC50 (концентрация полумаксимального ингибирования)
- CC50 (цитотоксическая концентрация)
- SI (индекс селективности)
"""

import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import KFold
from sklearn.preprocessing import PowerTransformer, QuantileTransformer
from sklearn.feature_selection import SelectKBest, mutual_info_regression
from sklearn.metrics import mean_squared_error
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.optimize import minimize
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor

warnings.filterwarnings('ignore')


class DataCleaner(BaseEstimator, TransformerMixin):
    """
    Класс для очистки данных:
    - Сохраняет медианы числовых колонок с пропусками
    - Создаёт индикаторы пропущенных значений
    - Заполняет пропуски медианами, бесконечности нулями
    """
    def fit(self, X, y=None):
        """Запоминает медианы для колонок с пропусками и их имена"""
        self.medians_ = {}
        self.missing_cols_ = []
        for col in X.select_dtypes(include=np.number).columns:
            if X[col].isna().any():
                self.medians_[col] = X[col].median()
                self.missing_cols_.append(col)
        return self
    
    def transform(self, X):
        """Применяет очистку: индикаторы пропусков, конвертация типов, заполнение NaN и inf"""
        X = X.copy()
        # Добавляем индикаторы пропущенных значений
        for col in self.missing_cols_:
            if col in X.columns:
                X[f'{col}_isnull'] = X[col].isna().astype(int)
        # Конвертируем все колонки в числовой формат
        for col in X.columns:
            X[col] = pd.to_numeric(X[col], errors='coerce')
        # Заполняем пропуски медианами
        for col, median_val in self.medians_.items():
            if col in X.columns:
                X[col].fillna(median_val, inplace=True)
        # Заменяем бесконечности на NaN, затем на 0
        X.replace([np.inf, -np.inf], np.nan, inplace=True)
        X.fillna(0.0, inplace=True)
        return X


class FeatureGenerator(BaseEstimator, TransformerMixin):
    """
    Генератор дополнительных признаков на основе физико-химических дескрипторов:
    - Создаёт 20+ инженерных признаков (отношения, произведения, степени)
    - Обрезает выбросы по квантилям на этапе fit
    """
    def __init__(self, clip_quantiles=(0.01, 0.99)):
        """Инициализация с границами квантилей для клиппирования выбросов"""
        self.clip_quantiles = clip_quantiles
        self.clip_bounds_ = {}
    
    def fit(self, X, y=None):
        """Вычисляет границы клиппирования для новых признаков по квантилям"""
        X_transformed = self._generate_features(X)
        for col in X_transformed.columns:
            if col not in X.columns:
                lower = X_transformed[col].quantile(self.clip_quantiles[0])
                upper = X_transformed[col].quantile(self.clip_quantiles[1])
                self.clip_bounds_[col] = (lower, upper)
        return self
    
    def transform(self, X):
        """Генерирует признаки и обрезает выбросы, заполняет оставшиеся NaN медианами"""
        X_out = self._generate_features(X)
        for col, (lower, upper) in self.clip_bounds_.items():
            if col in X_out.columns:
                X_out[col] = X_out[col].clip(lower, upper)
        X_out.replace([np.inf, -np.inf], np.nan, inplace=True)
        X_out.fillna(X_out.median(), inplace=True)
        return X_out
    
    def _generate_features(self, X):
        """
        Создаёт инженерные признаки из исходных дескрипторов:
        - Липофильность: LogP/TPSA, LipE, LogP²
        - Размер и полярность: MolWt*TPSA, TPSA/MolWt
        - Гибкость: RotBonds*LogP, RotBonds/HeavyAtoms
        - Сложность: BertzCT/HeavyAtoms, BertzCT*Rings
        - Водородные связи: HAcc+HDon, HAcc/HDon, HAcc*LogP
        - Заряд: Max-Min частичных зарядов, асимметрия
        - Комбинированные: QED*LogP, SP3 баланс, log MolWt
        """
        X_out = X.copy()
        # Безопасное получение колонок с заменой на 0 если отсутствуют
        get = lambda col: X_out.get(col, pd.Series(0.0, index=X_out.index)).fillna(0.0)
        
        # Исходные дескрипторы
        logp = get('MolLogP')
        tpsa = get('TPSA')
        molwt = get('MolWt')
        rings = get('RingCount')
        bertz = get('BertzCT')
        rot = get('NumRotatableBonds')
        h_acc = get('NumHAcceptors')
        h_don = get('NumHDonors')
        heavy = get('HeavyAtomCount')
        max_pc = get('MaxPartialCharge')
        min_pc = get('MinPartialCharge')
        qed = get('qed')
        sp3 = get('FractionCSP3')
        
        eps = 1e-8  # Малая константа для избегания деления на 0
        
        # Генерация признаков (20+ новых переменных)
        X_out['FE_LogP_TPSA'] = np.where(np.abs(tpsa) < eps, 0.0, logp / (tpsa + eps))
        X_out['FE_LipE'] = logp - tpsa / 100  # Эффективность липофильности
        X_out['FE_Size_Pol'] = molwt * tpsa / 1000  # Размер × полярность
        X_out['FE_Polar_Density'] = tpsa / (molwt + eps) * 100  # Плотность полярности
        X_out['FE_Flex_LogP'] = rot * logp / 10  # Гибкость × липофильность
        X_out['FE_Rotatable_Density'] = rot / (heavy + eps)  # Плотность вращаемых связей
        X_out['FE_Complexity'] = bertz / (heavy + eps)  # Сложность на атом
        X_out['FE_Aromatic_Frac'] = rings / (heavy + eps)  # Доля ароматических колец
        X_out['FE_Bertz_Rings'] = bertz * rings / 100  # Сложность × кольца
        X_out['FE_HBond_Total'] = h_acc + h_don  # Всего водородных связей
        X_out['FE_HBond_Density'] = (h_acc + h_don) / (heavy + eps)
        X_out['FE_HBond_Ratio'] = np.where(h_don < eps, 0.0, h_acc / (h_don + eps))  # Акцепторы/доноры
        X_out['FE_Charge_Span'] = max_pc - min_pc  # Размах заряда
        X_out['FE_Charge_Magnitude'] = np.abs(max_pc) + np.abs(min_pc)  # Магнитуда заряда
        X_out['FE_Charge_Asymmetry'] = np.where(np.abs(min_pc) < eps, 0.0, max_pc / (np.abs(min_pc) + eps))
        X_out['FE_SP3_Balance'] = sp3 * (1 - sp3)  # Баланс SP3 гибридизации
        X_out['FE_QED_LogP'] = qed * logp  # Drug-likeness × LogP
        X_out['FE_QED_TPSA'] = qed * tpsa
        X_out['FE_LogP_sq'] = np.sign(logp) * logp**2  # Квадрат LogP с сохранением знака
        X_out['FE_TPSA_sqrt'] = np.sqrt(np.abs(tpsa))  # Корень из TPSA
        X_out['FE_MW_log'] = np.log1p(molwt)  # Логарифм молекулярной массы
        X_out['FE_LogP_MW_ratio'] = logp / (np.log1p(molwt) + eps)  # LogP на логарифм массы
        X_out['FE_HAcc_LogP'] = h_acc * logp  # Акцепторы × LogP
        X_out['FE_HDon_LogP'] = h_don * logp  # Доноры × LogP
        X_out['FE_Rings_LogP'] = rings * logp  # Кольца × LogP
        
        return X_out


def train_target_model(train_df, target_name, test_df, feature_generator):
    """
    Обучение ансамбля моделей для одного целевого показателя.
    
    Этапы:
    1. Подготовка данных и логарифмирование целевой переменной
    2. 5-фолд кросс-валидация с обучением 4 типов моделей (XGBoost, LightGBM, ExtraTrees, GBR)
    3. Оптимизация весов ансамбля методом минимизации RMSE
    4. Финальное обучение на всех данных и предсказание на тесте
    
    Параметры:
    - train_df: обучающий DataFrame
    - target_name: имя целевой переменной ('IC50, mM', 'CC50, mM' или 'SI')
    - test_df: тестовый DataFrame
    - feature_generator: экземпляр FeatureGenerator
    
    Возвращает:
    - numpy array с предсказаниями для тестовой выборки
    """
    print(f"\n{'='*60}")
    print(f"TARGET: {target_name}")
    print(f"{'='*60}")
    
    # Подготовка обучающих данных
    df = train_df.copy()
    y_raw = df[target_name].copy()
    X = df.drop(columns=[target_name])
    
    # Удаляем другие целевые переменные из признаков
    other_targets = [c for c in ['IC50, mM', 'CC50, mM', 'SI'] if c != target_name and c in X.columns]
    if other_targets:
        X = X.drop(columns=other_targets)
    
    # Логарифмирование для концентраций (IC50, CC50)
    is_conc = target_name in ['IC50, mM', 'CC50, mM']
    y_log = np.log1p(y_raw)
    
    # Ограничение числа признаков
    k_features = min(200, X.shape[1])
    
    # Инициализация трансформеров
    cleaner = DataCleaner()
    scaler = QuantileTransformer(n_quantiles=100, output_distribution='normal', random_state=42)
    
    # Конфигурации моделей ансамбля
    models_config = {
        'XGB': {'n_estimators': 400, 'max_depth': 5, 'learning_rate': 0.03, 'subsample': 0.85, 
                'colsample_bytree': 0.85, 'reg_alpha': 0.1, 'reg_lambda': 1.0, 'random_state': 42, 'n_jobs': -1},
        'LGB': {'n_estimators': 400, 'max_depth': 5, 'learning_rate': 0.04, 'num_leaves': 31, 
                'subsample': 0.85, 'colsample_bytree': 0.85, 'reg_alpha': 0.05, 'reg_lambda': 0.5, 
                'random_state': 42, 'n_jobs': -1, 'verbose': -1},
        'ET': {'n_estimators': 350, 'max_depth': 10, 'min_samples_split': 10, 'min_samples_leaf': 5, 
               'random_state': 42, 'n_jobs': -1},
        'GBR': {'n_estimators': 350, 'max_depth': 5, 'learning_rate': 0.03, 'subsample': 0.85, 'random_state': 42}
    }
    
    # Кросс-валидация
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    oof_predictions = {name: np.zeros(len(X)) for name in models_config}
    
    print("Cross-validation...")
    for fold, (train_idx, val_idx) in enumerate(cv.split(X), 1):
        # Разделение на обучающую и валидационную выборки
        X_train_raw, X_val_raw = X.iloc[train_idx], X.iloc[val_idx]
        y_train_raw, y_val_raw = y_log.iloc[train_idx], y_log.iloc[val_idx]
        
        # Клиппирование целевой переменной для устранения выбросов
        lower_bound = max(0.1, y_train_raw.quantile(0.01))
        upper_bound = y_train_raw.quantile(0.99)
        y_train_clipped = y_train_raw.clip(lower_bound, upper_bound)
        
        # Power трансформация для нормализации распределения
        pt_fold = PowerTransformer(method='yeo-johnson', standardize=True)
        y_train_transformed = pt_fold.fit_transform(y_train_clipped.values.reshape(-1, 1)).ravel()
        y_val_clipped = y_val_raw.clip(lower_bound, upper_bound)
        y_val_transformed = pt_fold.transform(y_val_clipped.values.reshape(-1, 1)).ravel()
        
        # Пайплайн обработки признаков: очистка → генерация → масштабирование → отбор
        X_train_clean = cleaner.fit_transform(X_train_raw)
        X_val_clean = cleaner.transform(X_val_raw)
        X_train_feat = feature_generator.fit_transform(X_train_clean)
        X_val_feat = feature_generator.transform(X_val_clean)
        X_train_scaled = scaler.fit_transform(X_train_feat)
        X_val_scaled = scaler.transform(X_val_feat)
        
        # Отбор лучших признаков по взаимной информации
        selector = SelectKBest(mutual_info_regression, k=min(k_features, X_train_scaled.shape[1]))
        X_train_final = selector.fit_transform(X_train_scaled, y_train_transformed)
        X_val_final = selector.transform(X_val_scaled)
        
        # Обучение каждой модели ансамбля
        for name, config in models_config.items():
            if name == 'XGB':
                m = xgb.XGBRegressor(**config, early_stopping_rounds=50)
                m.fit(X_train_final, y_train_transformed, eval_set=[(X_val_final, y_val_transformed)], verbose=False)
            elif name == 'LGB':
                m = lgb.LGBMRegressor(**config)
                m.fit(X_train_final, y_train_transformed, eval_set=[(X_val_final, y_val_transformed)])
            elif name == 'GBR':
                m = GradientBoostingRegressor(**config, validation_fraction=0.1, n_iter_no_change=10)
                m.fit(X_train_final, y_train_transformed)
            else:
                m = ExtraTreesRegressor(**config)
                m.fit(X_train_final, y_train_transformed)
            
            # Обратное преобразование предсказаний в исходный масштаб
            y_pred_transformed = m.predict(X_val_final)
            y_pred = pt_fold.inverse_transform(y_pred_transformed.reshape(-1, 1)).ravel()
            y_pred_orig = np.expm1(y_pred) if is_conc else np.expm1(y_pred)
            oof_predictions[name][val_idx] = np.clip(y_pred_orig, 0.001, 10000)
        
        y_val_actual = np.expm1(y_val_raw) if is_conc else np.expm1(y_val_raw)
        rmse_fold = np.sqrt(mean_squared_error(y_val_actual, oof_predictions['LGB'][val_idx]))
        print(f"   Fold {fold}: RMSE = {rmse_fold:.4f}")
    
    # Оптимизация весов ансамбля
    print("Optimizing weights...")
    valid_models = [name for name in models_config if not np.any(np.isnan(oof_predictions[name]))]
    oof_matrix = np.column_stack([oof_predictions[name] for name in valid_models])
    y_actual = np.expm1(y_log) if is_conc else np.expm1(y_log)
    
    def objective(weights, oof, y_true, reg=0.01):
        """
        Целевая функция для оптимизации весов ансамбля.
        Минимизирует RMSE с L2-регуляризацией весов.
        """
        pred = np.dot(oof, weights / np.sum(weights))
        mse = np.mean((y_true - pred) ** 2)
        reg_term = reg * np.sum((weights - 1/len(weights))**2)
        return np.sqrt(mse) + reg_term
    
    # Инициализация равными весами, оптимизация с ограничениями
    init_w = np.ones(len(valid_models)) / len(valid_models)
    bounds = [(0.05, 0.95)] * len(valid_models)
    constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
    
    try:
        res = minimize(lambda w: objective(w, oof_matrix, y_actual), init_w, 
                      method='SLSQP', bounds=bounds, constraints=constraints)
        opt_weights = res.x / np.sum(res.x)
        # Если одна модель доминирует, возвращаем равные веса
        if max(opt_weights) > 0.5:
            opt_weights = np.ones(len(valid_models)) / len(valid_models)
    except:
        opt_weights = init_w
    
    print(f"   Weights: {dict(zip(valid_models, np.round(opt_weights, 3)))}")
    
    # Финальное обучение на всех данных
    print("Final training...")
    lower_bound_all = max(0.1, y_log.quantile(0.01))
    upper_bound_all = y_log.quantile(0.99)
    y_clipped_all = y_log.clip(lower_bound_all, upper_bound_all)
    
    pt_final = PowerTransformer(method='yeo-johnson', standardize=True)
    y_transformed_all = pt_final.fit_transform(y_clipped_all.values.reshape(-1, 1)).ravel()
    
    # Применяем все трансформации ко всему обучающему набору
    X_all_clean = cleaner.fit_transform(X)
    X_all_feat = feature_generator.fit_transform(X_all_clean)
    X_all_scaled = scaler.fit_transform(X_all_feat)
    
    final_selector = SelectKBest(mutual_info_regression, k=min(k_features, X_all_scaled.shape[1]))
    X_all_final = final_selector.fit_transform(X_all_scaled, y_transformed_all)
    
    # Обучаем финальные модели на полном датасете
    final_models = {}
    for name in valid_models:
        config = models_config[name]
        if name == 'XGB':
            m = xgb.XGBRegressor(**config)
            m.fit(X_all_final, y_transformed_all, verbose=False)
        elif name == 'LGB':
            m = lgb.LGBMRegressor(**config)
            m.fit(X_all_final, y_transformed_all)
        elif name == 'GBR':
            m = GradientBoostingRegressor(**config)
            m.fit(X_all_final, y_transformed_all)
        else:
            m = ExtraTreesRegressor(**config)
            m.fit(X_all_final, y_transformed_all)
        final_models[name] = m
    
    # Подготовка тестовых данных и предсказание
    print("Preparing test data...")
    X_test = test_df.copy()
    X_test_clean = cleaner.transform(X_test)
    X_test_feat = feature_generator.transform(X_test_clean)
    X_test_scaled = scaler.transform(X_test_feat)
    X_test_final = final_selector.transform(X_test_scaled)
    
    # Взвешенное усреднение предсказаний всех моделей
    predictions = []
    for name in valid_models:
        pred_transformed = final_models[name].predict(X_test_final)
        pred = pt_final.inverse_transform(pred_transformed.reshape(-1, 1)).ravel()
        pred_orig = np.expm1(pred) if is_conc else np.expm1(pred)
        predictions.append(np.clip(pred_orig, 0.001, 10000))
    
    pred_matrix = np.column_stack(predictions)
    final_predictions = np.dot(pred_matrix, opt_weights)
    
    return final_predictions


def post_process(predictions):
    """
    Постобработка предсказаний для обеспечения физической согласованности:
    1. Клиппирование значений в разумные диапазоны
    2. Обеспечение CC50 > IC50 (цитотоксичность должна быть выше противовирусной активности)
    3. Корректировка SI в соответствии с отношением CC50/IC50
    
    Параметры:
    - predictions: словарь с массивами предсказаний для IC50, CC50, SI
    
    Возвращает:
    - словарь с откорректированными предсказаниями
    """
    print("\n" + "="*80)
    print("POST-PROCESSING")
    print("="*80)
    
    ic50 = predictions['IC50, mM'].copy()
    cc50 = predictions['CC50, mM'].copy()
    si = predictions['SI'].copy()
    
    print(f"Before: IC50 [{ic50.min():.3f}, {ic50.max():.1f}] mean={ic50.mean():.2f}")
    print(f"Before: CC50 [{cc50.min():.3f}, {cc50.max():.1f}] mean={cc50.mean():.2f}")
    print(f"Before: SI   [{si.min():.3f}, {si.max():.1f}] mean={si.mean():.2f}")
    
    # Базовое клиппирование
    ic50 = np.clip(ic50, 0.001, 5000)
    cc50 = np.clip(cc50, 0.5, 10000)
    si = np.clip(si, 0.01, 10000)
    
    # Выявление и исправление физически невозможных случаев (CC50 < IC50)
    inconsistent = (cc50 < ic50) & (ic50 > 0)
    severe = inconsistent & (ic50 / np.clip(cc50, 0.1, None) > 2)  # Сильное нарушение
    moderate = inconsistent & ~severe  # Умеренное нарушение
    
    # Коррекция: CC50 не может быть меньше IC50
    if np.any(severe):
        cc50[severe] = ic50[severe] * 1.5
    if np.any(moderate):
        cc50[moderate] = np.maximum(cc50[moderate], ic50[moderate] * 0.9)
    
    # Коррекция SI на основе фактического отношения CC50/IC50
    calculated_si = cc50 / np.clip(ic50, 0.001, None)
    discrepancy = (np.abs(si - calculated_si) / np.clip(calculated_si, 0.01, None)) > 1.0
    # Сглаживание: 30% от предсказанного SI + 70% от рассчитанного
    si[discrepancy] = 0.3 * si[discrepancy] + 0.7 * calculated_si[discrepancy]
    si = np.clip(si, 0.01, 10000)
    
    print(f"After:  IC50 [{ic50.min():.3f}, {ic50.max():.1f}] mean={ic50.mean():.2f}")
    print(f"After:  CC50 [{cc50.min():.3f}, {cc50.max():.1f}] mean={cc50.mean():.2f}")
    print(f"After:  SI   [{si.min():.3f}, {si.max():.1f}] mean={si.mean():.2f}")
    
    predictions['IC50, mM'] = ic50
    predictions['CC50, mM'] = cc50
    predictions['SI'] = si
    
    return predictions


def main():
    """
    Главная функция пайплайна:
    1. Загрузка train.csv и test.csv
    2. Последовательное обучение для IC50, CC50, SI
    3. Постобработка для физической согласованности
    4. Сохранение submission.csv
    """
    print("="*80)
    print("MULTI-TARGET PREDICTION PIPELINE")
    print("="*80)
    
    # Загрузка данных
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')
    
    # Очистка имён колонок от пробелов
    train_df.columns = train_df.columns.str.strip()
    test_df.columns = test_df.columns.str.strip()
    
    # Сохранение индексов для submission
    indices = test_df['index'].values if 'index' in test_df.columns else test_df.index.values
    
    # Инициализация генератора признаков
    feature_generator = FeatureGenerator(clip_quantiles=(0.01, 0.99))
    
    # Последовательное обучение для каждой цели
    targets = ['IC50, mM', 'CC50, mM', 'SI']
    predictions = {}
    
    for target in targets:
        pred = train_target_model(train_df, target, test_df, feature_generator)
        if pred is not None:
            predictions[target] = pred
    
    # Постобработка для согласованности
    predictions = post_process(predictions)
    
    # Формирование и сохранение submission
    submission = pd.DataFrame({
        'index': indices,
        'IC50': predictions['IC50, mM'],
        'CC50': predictions['CC50, mM'],
        'SI': predictions['SI']
    })
    
    submission.to_csv('submission.csv', index=False)
    
    print("\n" + "="*80)
    print("RESULTS SAVED")
    print("="*80)
    print(f"IC50: [{submission['IC50'].min():.4f}, {submission['IC50'].max():.4f}] mean={submission['IC50'].mean():.4f}")
    print(f"CC50: [{submission['CC50'].min():.4f}, {submission['CC50'].max():.4f}] mean={submission['CC50'].mean():.4f}")
    print(f"SI:   [{submission['SI'].min():.4f}, {submission['SI'].max():.4f}] mean={submission['SI'].mean():.4f}")


if __name__ == "__main__":
    main()